**Big Data Analysis of Traffic Crash Factors**

This project analyzes a large traffic crash dataset to understand the various factors that contribute to accidents, injuries, and fatalities. It explores the influence of road conditions, weather, time patterns, and crash characteristics to identify key trends and predictors.

In [0]:
#Big Data Final Project Data Questions: 
    
#Ashley 
#Do roadways with traffic control devices have less crashes overall? 
#What are the top predictors that influence higher crash rates? 
#Are there any correlations between car damage amount and injuries? 

#Meet 
#Does weather condition impact crash type and fatalities? 
#Does weather/roadway condition have an impact on overall crashes? And out of those crashes how many are fatal? Provide top 5 arranging by the top number of fatalities. 
#What are the top causes of crashes overall and the total number of injuries associated with them? (include all relevant factors) 

#Patricia 
#Do traffic accidents tend to occur more frequently during specific months compared to others, and what are the weather conditions during these incidents?
#Accidents by month and weather condition
#Do crashes occur more frequently on weekdays or weekends? 
#What is the average number of vehicles involved in severe crashes?  
                                                                    
#Cody 
#What types of crashes result in the most injuries? (crash_type) 
#Are crashes more likely to occur at intersections or non-intersections 
#Which weather conditions contribute to the most severe injuries or fatalities and how much? 
#New Question 3: "How do different weather conditions contribute to the severity of crash injuries over time, and which conditions pose the highest risk of severe or fatal outcomes each month?"

**Common Python Libraries and Functions for Machine Learning and Data Processing**

This Python code imports key libraries for machine learning and big data. It covers data manipulation (NumPy, Pandas, PySpark), preprocessing (scikit-learn's imputation, scaling, encoding, text features), model building and evaluation (scikit-learn's classifiers/regressors, metrics, selection), and distributed computing with Spark.

In [0]:
# Python code for Machine learning: 
# Common imports
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error
from sklearn.tree import DecisionTreeRegressor
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, isnull, count, lower, sum, to_date, trim, avg, lit
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml import Pipeline
from pyspark.ml.regression import DecisionTreeRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import Imputer
from pyspark.sql.types import NumericType, StringType
from pyspark.sql import functions as F

from functools import reduce

**Loading and Registering Project Data with Spark**

- Specifies the file path (/FileStore/tables/final_project_big_data_2.csv) and type (csv).
- Configures CSV reading options: infers schema, uses the first row as header, and sets the delimiter to comma.
- Uses Spark to read the CSV file into a DataFrame (df).
- Creates a temporary SQL view named final_group_project_csv from the DataFrame.

In [0]:
# File location and type
file_location = "/FileStore/tables/final_project_big_data_2.csv"
file_type = "csv"

# CSV options
infer_schema = "true"
first_row_is_header = "true"
delimiter = ","

# The applied options are for CSV files. For other file types, these will be ignored.
df = spark.read.format(file_type) \
  .option("inferSchema", infer_schema) \
  .option("header", first_row_is_header) \
  .option("sep", delimiter) \
  .load(file_location)

#Display imported data
#display(df)

# Create a view or table

temp_table_name = "final_group_project_csv"

df.createOrReplaceTempView(temp_table_name)


**PySpark Data Cleaning and Preparation**

- Reads a CSV file into a Spark DataFrame named crashmaster.
- Performs ordinal encoding on the "damage" column, mapping string values to numerical categories.
- Filters the DataFrame to remove rows where string columns contain "unknown" or "other" (case-insensitive).
- Checks for missing data by calculating the sum of null values for each column.
- Creates a temporary SQL view named "cleaned_data" from the cleaned DataFrame.

In [0]:
from pyspark.sql.functions import when, trim, col, lower
from functools import reduce
from pyspark.sql import functions as F

df = spark.read.csv("/FileStore/shared_uploads/finalprojectgroup4040@gmail.com/export__1_.csv", header=True, inferSchema=True)
crashmaster = df

#Ordinal encoding
crashmaster = crashmaster.withColumn(
    "damage",
    trim(crashmaster["damage"])
).withColumn(
    "damage",
    when(col("damage") == "$500 OR LESS", 0)
    .when(col("damage") == "$501 - $1,500", 1)
    .when(col("damage") == "OVER $1,500", 2)
    .otherwise(None)
)


# List of string columns to check for 'unknown' or 'other'
string_cols = [col_name for col_name, dtype in crashmaster.dtypes if dtype == "string"]

# Build the filter condition for each unwanted value
unknown_filter = ~reduce(lambda x, y: x | y, [lower(col(c)).contains("unknown") for c in string_cols])
other_filter = ~reduce(lambda x, y: x | y, [lower(col(c)).contains("other") for c in string_cols])

# Apply filters
crashmaster = crashmaster.filter(unknown_filter & other_filter)

# Check for missing data
from pyspark.sql import functions as F

crashmaster.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in crashmaster.columns
 ]).toPandas().T

# Store the 'crashmaster' DataFrame into a table named 'cleaned_data'
crashmaster.createOrReplaceTempView("cleaned_data") #create temp view



**SQL Queries and Exploratory Analysis**


**Top Weather and Road Surface Conditions Linked to Fatal Crashes**

This SQL query investigates the impact of weather and road surface conditions on traffic crashes and fatalities. It extracts data from the cleaned_data table, grouping by weather and road surface conditions. For each combination, it calculates the total number of crashes (Overall Crashes) and the total number of fatalities (Overall Fatalities). The results are sorted by the highest number of fatalities and limited to the top 5 combinations. This helps identify the most dangerous weather and roadway conditions associated with fatal crashes.

**Visualization Explanation**

This grouped bar chart visualizes the average number of overall crashes and fatalities across different combinations of weather and road surface conditions. The x-axis represents three road surface types: Dry, Snow or Slush, and Wet, while each color-coded bar corresponds to a specific weather condition (e.g., Clear, Cloudy/Overcast, Rain).

**Key insights:**

Dry & Clear conditions see the highest volume of crashes (around 30,000) and fatalities (249 on average), likely due to higher traffic levels.

Rain & Wet conditions have fewer crashes (16,264) but relatively high fatalities (31), suggesting a higher severity when accidents do occur.

Snow or Slush conditions are associated with lower crash counts (e.g., 1,017 for Clear weather) and fatalities (3), possibly due to reduced driving activity or increased caution.

In [0]:
#Does weather/roadway condition have an impact on overall crashes? And out of those crashes how many are fatal? Provide top 5 arranging by the top number of fatalities.

# Execute the SQL query and store the result in a DataFrame
df = spark.sql("""
    SELECT
        DISTINCT roadway_surface_cond AS `Road Surface Condition`,
        weather_condition AS `Weather Condition`,
        COUNT(*) AS `Overall Crashes`,
        SUM(injuries_fatal) AS `Overall Fatalities`
    FROM cleaned_data
    GROUP BY `Weather Condition`, `Road Surface Condition`
    ORDER BY `Overall Fatalities` DESC
    LIMIT 5
""")

display(df)

Road Surface Condition,Weather Condition,Overall Crashes,Overall Fatalities
DRY,CLEAR,126233,249
WET,RAIN,16264,31
DRY,CLOUDY/OVERCAST,4053,8
WET,CLEAR,5521,6
SNOW OR SLUSH,CLEAR,1017,3


Databricks visualization. Run in Databricks to view.


**Top Weather-Related Crash Types by Fatality Count**

This SQL query evaluates how different weather conditions relate to various crash types and their associated fatalities. By grouping crash records by both weather_condition and crash_type, it calculates:

The total number of crashes (Overall_Crash)

The total number of fatalities (Overall_Fatalities)

It then ranks the combinations by the highest number of fatalities and returns the top 5 most deadly weather-crash type scenarios. This analysis helps highlight which crash types under specific weather conditions are most fatal.

**Visualization Description**

This bar chart illustrates the relationship between weather conditions and the number of injury/tow-related crashes and their associated fatalities. The x-axis shows various weather conditions, while the bars represent the average number of crashes (blue) and average fatalities (red) for each condition.

**Key Findings:**

Clear weather has the highest crash count (~15,000) and also the highest average fatalities (259). This likely reflects higher traffic volumes rather than hazardous conditions.

Rain shows a moderate crash count (~8,300) but a relatively high fatality count (31), suggesting increased severity during wet weather.

Cloudy/Overcast and Snow conditions have lower crash and fatality rates, while Severe Crosswind Gate conditions appear rarely, with minimal crash and fatality occurrences.

This visualization supports the conclusion that while most crashes occur during clear weather, rain poses a greater risk of fatal outcomes relative to total crashes.

In [0]:
%sql

--Does weather condition impact crash type and fatalities? 

SELECT weather_condition AS Weather_Condition, crash_type AS Crash_Type,
COUNT(*) AS Overall_Crash,
SUM(injuries_fatal) AS Overall_Fatalities
FROM cleaned_data
GROUP BY Weather_Condition, Crash_Type
ORDER BY Overall_Fatalities DESC
LIMIT 5;

Weather_Condition,Crash_Type,Overall_Crash,Overall_Fatalities
CLEAR,INJURY AND / OR TOW DUE TO CRASH,59710,259
RAIN,INJURY AND / OR TOW DUE TO CRASH,8335,31
CLOUDY/OVERCAST,INJURY AND / OR TOW DUE TO CRASH,2796,10
SNOW,INJURY AND / OR TOW DUE TO CRASH,2221,3
SEVERE CROSS WIND GATE,INJURY AND / OR TOW DUE TO CRASH,11,1


Databricks visualization. Run in Databricks to view.

**Top Crash Scenarios Linked to the Highest Injury Counts**

This SQL query investigates how different types of traffic control devices relate to the overall number of crashes. By grouping crash records by the traffic_control_device, it calculates:

The total number of crashes (total_crashes)

It then ranks the devices by the highest number of crashes and returns a list sorted in descending order.

This analysis helps identify which traffic control devices are most frequently associated with crashes and provides insight into potential areas where traffic safety measures may need to be reassessed or strengthened.



In [0]:
%sql

--What are the top causes of crashes overall and the total number of injuries associated with them? (include all relevant factors)

SELECT traffic_control_device AS `Traffic Device`,
weather_condition AS `Weather Condition`,
lighting_condition AS `Lighting Condition`,
first_crash_type AS `First Crash Type`,
trafficway_type AS `Road Type`,
alignment AS `Allignment`,
roadway_surface_cond AS `Road Surface Condition`,
road_defect AS `Road Defects`,
crash_type AS `Crash Type`,
prim_contributory_cause AS `Primary Cause`,
COUNT(*) AS `Total Number Of Injuries`
FROM cleaned_data
GROUP BY `Traffic Device`, `Weather Condition`, `Lighting Condition`, `First Crash Type`, `Road Type`, `Allignment`, `Road Surface Condition`, `Road Defects`, `Crash Type`, `Primary Cause`
ORDER BY `Total Number Of Injuries` DESC
LIMIT 1

Traffic Device,Weather Condition,Lighting Condition,First Crash Type,Road Type,Allignment,Road Surface Condition,Road Defects,Crash Type,Primary Cause,Total Number Of Injuries
TRAFFIC SIGNAL,CLEAR,DAYLIGHT,REAR END,NOT DIVIDED,STRAIGHT AND LEVEL,DRY,NO DEFECTS,NO INJURY / DRIVE AWAY,FOLLOWING TOO CLOSELY,2173



**Crash Volume by Traffic Control Device Presence**

This SQL query explores whether the presence of different traffic control devices is associated with a higher or lower frequency of overall crashes. By grouping crash records by traffic_control_device, it calculates:

The total number of crashes (total_crashes)

It then ranks the devices by total crash count in descending order to identify which types are linked to the most crash occurrences.

This analysis helps reveal which traffic control mechanisms—such as stop signs, signals, or lack of control—are most commonly present at crash locations, providing insight into their potential effectiveness or exposure levels.


**Visualization Description**

This horizontal bar chart displays the total number of crashes associated with various traffic control devices.

**Key Findings:**

Traffic signals are linked to the highest number of crashes (over 43,000), followed by stop signs/flashers (around 39,000).

Roadways with no control devices still account for a significant number of crashes (24,147), indicating elevated risk in uncontrolled areas.

Other devices such as pedestrian crossing signs, yield signs, and flashing control signals are associated with far fewer crash incidents.

This visualization suggests that although intersections with signals and stop signs experience high crash volumes, likely due to higher traffic density—uncontrolled roadways still represent a major share of total crashes, which could warrant improved infrastructure or regulation.

In [0]:
%sql
--#Do roadways with traffic control devices have less crashes overall? 
SELECT traffic_control_device, 
COUNT(*) AS total_crashes
FROM cleaned_data
GROUP BY traffic_control_device
ORDER BY total_crashes DESC;


traffic_control_device,total_crashes
TRAFFIC SIGNAL,98198
STOP SIGN/FLASHER,39000
NO CONTROLS,24147
YIELD,368
PEDESTRIAN CROSSING SIGN,189
LANE USE MARKING,122
FLASHING CONTROL SIGNAL,106
POLICE/FLAGMAN,78
RAILROAD CROSSING GATE,58
SCHOOL ZONE,19


Databricks visualization. Run in Databricks to view.


**Comparing Crash Rates on Controlled vs. Uncontrolled Roadways**

This SQL script contains two complementary queries to evaluate how the presence of traffic control devices impacts crash frequency and distribution.

**The first query:**

Groups roadways into two categories:

'No Control' for null or 'None' traffic control devices

'With Control' for all others

Calculates the total number of crashes (total_crashes) for each category

**The second query:**

Calculates the total number of crashes per traffic control device type

Computes the percentage of total crashes each device type represents (pct_of_total)

Filters out nulls and ranks results by crash volume

This dual-query approach offers both a broad comparison of controlled vs. uncontrolled roads and a granular breakdown of which control types are most frequently involved in crashes, allowing for more targeted safety recommendations.

In [0]:
%sql

--#Comparing roadways with vs without traffic control devices
SELECT 
    CASE 
        WHEN traffic_control_device IS NULL OR traffic_control_device = 'None' THEN 'No Control'
        ELSE 'With Control'
    END AS control_status,
    COUNT(*) AS total_crashes
FROM cleaned_data
GROUP BY control_status;

-- Adding total percentage and filtering out nulls for cleaner results
SELECT 
  traffic_control_device,
  COUNT(*) AS total_crashes,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_of_total
FROM cleaned_data
WHERE traffic_control_device IS NOT NULL
GROUP BY traffic_control_device
ORDER BY total_crashes DESC;



traffic_control_device,total_crashes,pct_of_total
TRAFFIC SIGNAL,98198,60.50
STOP SIGN/FLASHER,39000,24.03
NO CONTROLS,24147,14.88
YIELD,368,0.23
PEDESTRIAN CROSSING SIGN,189,0.12
LANE USE MARKING,122,0.08
FLASHING CONTROL SIGNAL,106,0.07
POLICE/FLAGMAN,78,0.05
RAILROAD CROSSING GATE,58,0.04
SCHOOL ZONE,19,0.01


**Analyzing the Relationship Between Car Damage Levels and Injury Severity**


This SQL script investigates whether there's a meaningful relationship between vehicle damage levels and the severity or frequency of injuries resulting from crashes. It does this through four complementary queries:

Crash Frequency by Damage and Injury Count

Groups records by damage and injuries_total

Counts total crashes per damage-injury combination

Average Injuries per Damage Level

Calculates the mean number of injuries (avg_injuries) for each damage level

Ranks results from most to least severe

Crash Share and Injury Severity by Damage Level

Adds the percentage of total crashes per damage level

Includes average injuries for context on both frequency and impact

Categorical Damage Analysis

Reclassifies numeric damage values into categories (Low, Medium, High, Unknown)

Calculates and ranks average injuries for each category

This set of queries provides a comprehensive look at whether greater car damage corresponds to more severe injuries, offering insight into crash outcomes based on impact severity.


**Visualization Description:**

This bar chart illustrates the relationship between car damage severity (categorized as Low, Medium, and High) and the average number of injuries sustained per crash.

**Key Findings:**

High damage crashes result in the highest average injuries (~0.46), indicating a strong correlation between severe vehicle damage and injury severity.

Surprisingly, Low damage crashes still show a relatively high average injury count (~0.39), suggesting that even minor vehicle damage can result in physical harm.

Medium damage crashes have the lowest average injury count (~0.16), which may indicate inconsistencies in reporting or classification.

This chart supports the idea that while more extensive vehicle damage tends to correlate with more injuries, lower damage doesn’t necessarily imply low risk.

In [0]:
%sql
--Are there any correlations between car damage amount and injuries?
SELECT 
  damage
  injuries_total,
  COUNT(*) AS total_crashes
FROM cleaned_data
GROUP BY damage, injuries_total
ORDER BY damage, total_crashes DESC;

--#Average injuries per damage amount
SELECT  
  damage,
  AVG(injuries_total) AS avg_injuries
FROM cleaned_data
GROUP BY damage
ORDER BY avg_injuries DESC;

-- Add percent of total crashes and avg injuries per damage level
SELECT 
  damage,
  COUNT(*) AS crash_count,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_of_total,
  ROUND(AVG(injuries_total), 2) AS avg_injuries
FROM cleaned_data
WHERE damage IS NOT NULL
GROUP BY damage
ORDER BY avg_injuries DESC;

SELECT 
  CASE 
    WHEN damage = 0 THEN 'Low'
    WHEN damage = 1 THEN 'Medium'
    WHEN damage = 2 THEN 'High'
    ELSE 'Unknown'
  END AS damage,
  ROUND(AVG(injuries_total), 2) AS avg_injuries
FROM cleaned_data
GROUP BY damage
ORDER BY avg_injuries DESC;




damage,avg_injuries
High,0.46
Low,0.39
Medium,0.16


Databricks visualization. Run in Databricks to view.


**Monthly Crash Frequency and Dominant Weather Conditions**

This SQL query analyzes whether traffic accidents are more frequent in certain months and identifies the most common weather condition during those incidents.

The logic is executed in two main steps:

**Monthly Crash Totals:**

Aggregates the total number of crashes (total_accidents) by crash_month

Converts month numbers into month names (e.g., 1 → January)

**Most Common Weather per Month:**

Counts occurrences of each weather condition per month

Uses a window function to rank conditions and select the most frequent weather type for each month

**Finally, it joins both datasets to display, for each month:**

The total number of crashes

The most frequent weather condition during those crashes

The count of crashes under that weather condition

This query provides valuable insights for seasonal trend analysis and helps correlate weather conditions with peak accident months.

**Visualization Description:**

This horizontal bar chart displays the average number of crashes occurring each month under clear weather conditions, providing insight into seasonal traffic patterns.

**Key Findings:**

September shows the highest number of crashes under clear weather (13,186), followed closely by August (12,986) and July (12,702).

The lowest crash counts under clear weather occur in February (8,524) and January (8,873).

Overall, clear conditions dominate across all months, suggesting that a high volume of crashes occurs regardless of weather visibility, likely due to increased traffic during fair weather periods.

This visualization supports the idea that seasonal traffic volume, rather than adverse weather, may be a stronger driver of crash frequency in many months.

In [0]:
%sql
--Do traffic accidents tend to occur more frequently during specific months compared to others,
 --and what are the weather conditions during these incidents?

SELECT
    CASE t1.crash_month
        WHEN 1 THEN 'January'
        WHEN 2 THEN 'February'
        WHEN 3 THEN 'March'
        WHEN 4 THEN 'April'
        WHEN 5 THEN 'May'
        WHEN 6 THEN 'June'
        WHEN 7 THEN 'July'
        WHEN 8 THEN 'August'
        WHEN 9 THEN 'September'
        WHEN 10 THEN 'October'
        WHEN 11 THEN 'November'
        WHEN 12 THEN 'December'
    END AS month_name,
    t2.weather_condition,
    t2.weather_count
FROM (
    SELECT
        crash_month,
        COUNT(*) AS total_accidents
    FROM cleaned_data
    GROUP BY crash_month
) t1
JOIN (
    SELECT
        crash_month,
        weather_condition,
        COUNT(*) AS weather_count,
        ROW_NUMBER() OVER(PARTITION BY crash_month ORDER BY COUNT(*) DESC) as rn
    FROM cleaned_data
    GROUP BY crash_month, weather_condition
) t2 ON t1.crash_month = t2.crash_month AND t2.rn = 1
ORDER BY t1.crash_month;

month_name,weather_condition,weather_count
January,CLEAR,8873
February,CLEAR,8524
March,CLEAR,9590
April,CLEAR,9229
May,CLEAR,11407
June,CLEAR,12361
July,CLEAR,12702
August,CLEAR,12986
September,CLEAR,13186
October,CLEAR,12254


Databricks visualization. Run in Databricks to view.

**Monthly Weather Trends and Crash Volume**

This SQL query explores how crash frequency fluctuates across different weather conditions and months of the year.

Query Logic:
Converts numeric crash_month values into month names (e.g., 1 → January).

Groups data by both month_name and weather_condition.

Counts the total number of crashes (total_accidents) for each weather condition per month.

Orders the output by month and crash count in descending order.

**Visualization Description**

**Key Findings:**

Clear weather consistently results in the highest number of crashes across all months, indicating that volume—not visibility—may be a stronger crash driver.

Rain and snow are the next most frequent contributors, particularly in colder months (e.g., January, February, November).

Severe or rare weather events like freezing rain, blowing snow, and fog contribute to fewer crashes but may still pose significant risk due to lower visibility and road hazards.

September and August show the highest total crash counts in clear weather, with 13,186 and 12,986 crashes respectively.

This monthly breakdown can help transportation planners and public safety officials prepare season-specific safety interventions and messaging.


In [0]:
%sql
-- 3. Accidents by month and weather condition
SELECT 
    CASE crash_month
        WHEN 1 THEN 'January'
        WHEN 2 THEN 'February'
        WHEN 3 THEN 'March'
        WHEN 4 THEN 'April'
        WHEN 5 THEN 'May'
        WHEN 6 THEN 'June'
        WHEN 7 THEN 'July'
        WHEN 8 THEN 'August'
        WHEN 9 THEN 'September'
        WHEN 10 THEN 'October'
        WHEN 11 THEN 'November'
        WHEN 12 THEN 'December'
    END AS month_name,
    weather_condition,
    COUNT(*) AS total_accidents
FROM cleaned_data
GROUP BY crash_month, weather_condition
ORDER BY crash_month, total_accidents DESC;



month_name,weather_condition,total_accidents
January,CLEAR,8873
January,SNOW,1789
January,RAIN,928
January,CLOUDY/OVERCAST,678
January,FREEZING RAIN/DRIZZLE,143
January,FOG/SMOKE/HAZE,80
January,SLEET/HAIL,75
January,BLOWING SNOW,24
January,SEVERE CROSS WIND GATE,2
February,CLEAR,8524


Databricks visualization. Run in Databricks to view.

**Crash Frequency: Weekday vs. Weekend**

This SQL query analyzes crash occurrences based on whether they happened during the weekend or weekday.

Query Logic:
Categorizes each crash as occurring on a "Weekday" (Monday through Friday) or "Weekend" (Saturday and Sunday) using crash_day_of_week

Aggregates the total number of crashes in each category

Orders the result by total crash count in descending order

**Visualization Description**

**Key Insights:**

Weekday crashes occur nearly 3 times more often than weekend crashes.

This trend may be driven by higher traffic volume during workweek commutes and daytime activity patterns.

In [0]:
%sql
SELECT
    CASE
        WHEN crash_day_of_week IN (1, 7) THEN 'Weekend'
        ELSE 'Weekday'
    END AS day_type,
    COUNT(*) AS crash_count
FROM
    cleaned_data
GROUP BY
    day_type
ORDER BY
    crash_count DESC;

day_type,crash_count
Weekday,119188
Weekend,43132


Databricks visualization. Run in Databricks to view.

**Average Number of Vehicles Involved in Severe Crashes**

This SQL query calculates the average number of vehicles involved in crashes that result in the most serious outcomes—incapacitating injuries or fatalities.

Query Logic:
Filters the dataset to only include crashes where most_severe_injury is either "INCAPACITATING INJURY" or "FATAL INJURY"

Calculates the average number of vehicles involved (num_units) in those crashes

In [0]:
%sql

SELECT
    AVG(num_units * 1.0) AS avg_vehicles_in_severe_crashes
FROM
    cleaned_data
WHERE
    most_severe_injury IN ('INCAPACITATING INJURY', 'FATAL INJURY');

avg_vehicles_in_severe_crashes
2.18754


**Crash Types and Their Relationship to Injury Frequency**

This SQL query explores which crash types are most commonly associated with injuries, offering insight into the severity of different crash classifications.

Query Logic:
Groups the dataset by the crash_type column

Counts the number of crash incidents (Total Injuries) for each type

Orders the results in descending order to identify which crash types most frequently involve injuries

In [0]:
%sql
-- What types of crashes result in the most injuries? (crash_type) 
SELECT crash_type AS `Crash Type`, 
COUNT(*) AS `Total Injuries` 
FROM cleaned_data 
GROUP BY crash_type 
ORDER BY "Total Injuries" DESC;

Crash Type,Total Injuries
INJURY AND / OR TOW DUE TO CRASH,73617
NO INJURY / DRIVE AWAY,88703


**Crash Likelihood at Intersections vs. Non-Intersections**

This SQL query investigates where crashes are more likely to occur—at intersections or elsewhere on the roadway.

Query Logic:
Categorizes each crash record using a CASE statement:

'INTERSECTION' if trafficway_type contains the word “INTERSECTION”

'Others' for all other road types

Counts the number of crash records for each category

Groups and orders the results by location type

**Visualization Description**

**Key Insights:**

Crashes on non-intersection roadways ("Others") account for over 95% of all reported crashes in the dataset.

Intersection-related crashes represent a relatively small share (~5%), but may still involve more complex scenarios like turning or cross-traffic.

These findings highlight that while intersections are high-conflict zones, non-intersection roadways contribute the overwhelming majority of crashes and should be a focus for safety and infrastructure improvements.

In [0]:
%sql
--Are crashes more likely to occur at intersections or non-intersections 
SELECT 

CASE 
WHEN trafficway_type LIKE '%INTERSECTION%' THEN 'INTERSECTION'
ELSE 'Others'
END AS `Intersection_Type`,

COUNT(*) AS `Total`
FROM cleaned_data

GROUP BY CASE 
WHEN trafficway_type LIKE '%INTERSECTION%' THEN 'INTERSECTION'
ELSE 'Others'
END;

Intersection_Type,Total
INTERSECTION,7826
Others,154494


Databricks visualization. Run in Databricks to view.

**Weather Conditions and Their Impact on Severe Crash Injuries**

This SQL query investigates how different weather conditions correlate with the most severe crash outcomes—fatalities and incapacitating injuries.

The script performs the following analysis:

Filters the dataset to only include crashes where the most severe injury is either “FATAL” or “INCAPACITATING INJURY.”

Groups results by weather condition and classifies the injury type as "Severe Injury/Fatal."

Counts the total number of such severe incidents per weather condition.

Orders the results by descending crash volume to identify the most dangerous weather scenarios.

In [0]:
%sql
--Which weather condition contributes to the most severe injuries or fatalities and how much? 
SELECT 
    CASE 
        WHEN most_severe_injury LIKE 'INCAPACITATING INJURY' OR most_severe_injury LIKE 'FATAL' THEN 'Severe Injury/Fatal'
    END AS `Injury_Type`,
    weather_condition,
    COUNT(*) AS `Total`
FROM cleaned_data
WHERE most_severe_injury LIKE 'INCAPACITATING INJURY' 
   OR most_severe_injury LIKE 'FATAL'
GROUP BY 
    CASE 
        WHEN most_severe_injury LIKE 'INCAPACITATING INJURY' OR most_severe_injury LIKE 'FATAL' THEN 'Severe Injury/Fatal'
    END,
    weather_condition
ORDER BY `Total` DESC;

Injury_Type,weather_condition,Total
Severe Injury/Fatal,CLEAR,4654
Severe Injury/Fatal,RAIN,560
Severe Injury/Fatal,CLOUDY/OVERCAST,181
Severe Injury/Fatal,SNOW,111
Severe Injury/Fatal,FREEZING RAIN/DRIZZLE,15
Severe Injury/Fatal,SLEET/HAIL,10
Severe Injury/Fatal,FOG/SMOKE/HAZE,6
Severe Injury/Fatal,BLOWING SNOW,4
Severe Injury/Fatal,SEVERE CROSS WIND GATE,1


**Weather Conditions and Injury Severity Distribution by Month**

This SQL analysis evaluates how different weather conditions contribute to crash injury severity over time, highlighting those with the highest monthly impact.

This analysis helps identify which weather conditions are most frequently associated with different injury severities, month by month—useful for seasonal crash risk mitigation and public safety campaigns.


In [0]:
%sql
-- How do different weather conditions contribute to the severity of crash injuries over time, and which conditions pose the highest risk of severe or fatal outcomes each month?
WITH categorized_injuries AS (
  SELECT 
    weather_condition,
    crash_month,
    CASE 
      WHEN most_severe_injury IN ('FATAL', 'INCAPACITATING INJURY') THEN 'Severe Injury/Fatal'
      WHEN most_severe_injury IN ('NONINCAPACITATING INJURY', 'REPORTED, NOT EVIDENT') THEN 'Less Severe Injury'
      ELSE 'No/Unknown Injury'
    END AS injury_category
  FROM cleaned_data
  WHERE weather_condition IS NOT NULL
)

SELECT 
  weather_condition,
  crash_month,
  injury_category,
  COUNT(*) AS total_incidents,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY crash_month), 2) AS percentage_of_month,
  RANK() OVER (PARTITION BY crash_month ORDER BY COUNT(*) DESC) AS rank_in_month
FROM categorized_injuries
GROUP BY weather_condition, crash_month, injury_category
ORDER BY crash_month, rank_in_month
LIMIT 5;


weather_condition,crash_month,injury_category,total_incidents,percentage_of_month,rank_in_month
CLEAR,1,No/Unknown Injury,6674,53.00,1
CLEAR,1,Less Severe Injury,1911,15.18,2
SNOW,1,No/Unknown Injury,1379,10.95,3
RAIN,1,No/Unknown Injury,657,5.22,4
CLOUDY/OVERCAST,1,No/Unknown Injury,519,4.12,5


### **Machine Learning:**

**Data Preparation:**

- Prepares data to identify predictors of crash injuries using the crashmaster DataFrame.
- Splits data (70% training, 30% testing) and defines injuries_total as the target.
- Separates features and identifies numeric/categorical columns.

**Feature Processing (Pipeline):**

- Imputes, assembles, and scales numeric columns.
- Indexes and one-hot encodes categorical columns.
- Combines features into a single column.
- Fits pipeline to training data, transforms both sets.

**Baseline Model Evaluation:**

**Dummy Model:** Predicts mean injuries_total (RMSE train: 0.8093, test: 0.8121).

**Interpretation:** Dummy Model RMSE of ~0.81 is the baseline for other models.

**Decision Tree Model Evaluation**

**Model** A Decision Tree Regressor with a maximum depth of 3 was trained on the transformed training data. The model was then used to make predictions on both the training and test sets. The Root Mean Squared Error (RMSE) was computed for evaluation, resulting in:

Train RMSE: 0.3068
Test RMSE: 0.3034

**Interpretation:** This indicates the model performs similarly on both sets, suggesting minimal overfitting.

In [0]:
# What types of crashes result in the most injuries?
#Pull cleaned data 
crash = crashmaster

# Now you can split it
train_set, test_set = crash.randomSplit([0.7, 0.3], seed=42)

# Target and input split, Trying to find the predictors for injuries.
# Target
label_col = "injuries_total"

# Separate features
train_inputs = train_set.drop(label_col)
test_inputs = test_set.drop(label_col)

In [0]:
# Find numeric columns
numeric_columns = [field.name for field in train_inputs.schema.fields if isinstance(field.dataType, NumericType)]

# Find categorical (string) columns
categorical_columns = [field.name for field in train_inputs.schema.fields if isinstance(field.dataType, StringType)]

# Print
print("Numeric columns are:", numeric_columns)
print("Categorical columns are:", categorical_columns)

Numeric columns are: ['crash_date', 'damage', 'num_units', 'injuries_fatal', 'injuries_incapacitating', 'injuries_non_incapacitating', 'injuries_reported_not_evident', 'injuries_no_indication', 'crash_hour', 'crash_day_of_week', 'crash_month']
Categorical columns are: ['traffic_control_device', 'weather_condition', 'lighting_condition', 'first_crash_type', 'trafficway_type', 'alignment', 'roadway_surface_cond', 'road_defect', 'crash_type', 'intersection_related_i', 'prim_contributory_cause', 'most_severe_injury']


In [0]:
# Numeric Transformer (Imputer for any missing values, and scalar)
imputer = Imputer(inputCols=numeric_columns, outputCols=[f"{col}_imputed" for col in numeric_columns])
assembler_numeric = VectorAssembler(inputCols=[f"{col}_imputed" for col in numeric_columns], outputCol="numeric_features")
scaler = StandardScaler(inputCol="numeric_features", outputCol="scaled_numeric_features")


# Create categorical pipeline
indexers = [StringIndexer(inputCol=col, outputCol=f"{col}_indexed", handleInvalid="keep") for col in categorical_columns]
encoders = [OneHotEncoder(inputCol=f"{col}_indexed", outputCol=f"{col}_encoded") for col in categorical_columns]

# Final features
feature_cols = ["scaled_numeric_features"] + [f"{col}_encoded" for col in categorical_columns]
assembler_final = VectorAssembler(inputCols=feature_cols, outputCol="features")

# Build pipeline
stages = [imputer, assembler_numeric, scaler] + indexers + encoders + [assembler_final]
pipeline = Pipeline(stages=stages)

In [0]:
# Fit and transform the train data
model = pipeline.fit(train_set)
train_pred = model.transform(train_set)

# Apply the pipeline to the test set
test_pred = model.transform(test_set)

# Show the transformed results
#train_transformed.show()
#test_transformed.show()

In [0]:
# Find the Baseline
# Dummy Model (Predict Mean)
# Compute mean
label_mean = train_set.select(avg(label_col)).first()[0]

# Predict constant mean
train_dummy = train_pred.withColumn("prediction", lit(label_mean))
test_dummy = test_pred.withColumn("prediction", lit(label_mean))

# Evaluate
evaluator = RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="rmse")

train_rmse = evaluator.evaluate(train_dummy)
test_rmse = evaluator.evaluate(test_dummy)

print(f"Train RMSE (Dummy Model): {train_rmse}")
print(f"Test RMSE (Dummy Model): {test_rmse}")

Train RMSE (Dummy Model): 0.8093467658551238
Test RMSE (Dummy Model): 0.8120982531418692


In [0]:
# Decision Tree Model
# Now fit decision tree on transformed train data
tree = DecisionTreeRegressor(featuresCol="features", labelCol=label_col, maxDepth=3)
tree_model = tree.fit(train_pred)

# Predict
train_tree_pred = tree_model.transform(train_pred)
test_tree_pred = tree_model.transform(test_pred)

# Evaluate
train_rmse_tree = evaluator.evaluate(train_tree_pred)
test_rmse_tree = evaluator.evaluate(test_tree_pred)

print(f"Train RMSE (Decision Tree): {train_rmse_tree}")
print(f"Test RMSE (Decision Tree): {test_rmse_tree}")

Train RMSE (Decision Tree): 0.30680925180215596
Test RMSE (Decision Tree): 0.30339171753897226


## ML Model 2

**Data Preparation:**

- Creates a binary injury_flag column (1 for injury, 0 for no injury).
- Engineers features like night_crash, weekend_crash, rush_hour_crash, multi_vehicle, any_incapacitating, and any_fatalities.
- Drops original columns (injuries_total, etc.).
- Splits the data into training (70%) and testing (30%) sets.
- Identifies categorical and numeric columns.

**Feature Processing (Pipeline):**

- Indexes and one-hot encodes categorical features.
- Fills null values in numeric columns with 0.
- Assembles features into a single vector column named "features_raw".
- Scales the features using StandardScaler.

**Model Training:**

- Trains a Logistic Regression model on the training data.
- Prediction and Evaluation:
- Makes predictions on both the training and testing sets.
- Evaluates the model's accuracy using MulticlassClassificationEvaluator.
- Prints the training and testing accuracies.

**Results** 

Training Accuracy: 0.7673
Test Accuracy: 0.7670

This means the model is able to correctly classify about 76.7% of the observations in both the training and testing datasets.

In [0]:

# Start Spark session if needed
spark = SparkSession.builder.getOrCreate()

# Assume crashmaster is your cleaned Spark DataFrame
crash = crashmaster

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when 

# Binary encoding: injury vs no injury
crash = crash.withColumn("injury_flag", when(col("injuries_total") > 0, 1).otherwise(0))

# Feature Engineering
crash = crash.withColumn("night_crash", when((col("crash_hour") >= 20) | (col("crash_hour") <= 6), 1).otherwise(0)) \
.withColumn("weekend_crash", when(col("crash_day_of_week").isin(6, 7), 1).otherwise(0)) \
.withColumn("rush_hour_crash", when(((col("crash_hour") >= 7) & (col("crash_hour") <= 9)) |
((col("crash_hour") >= 16) & (col("crash_hour") <= 18)), 1).otherwise(0)) \
.withColumn("multi_vehicle", when(col("num_units") > 1, 1).otherwise(0)) \
.withColumn("any_incapacitating", when(col("injuries_incapacitating") > 0, 1).otherwise(0)) \
.withColumn("any_fatalities", when(col("injuries_fatal") > 0, 1).otherwise(0))

# Drop unnecessary columns
columns_to_drop = [
'injuries_total', 'crash_date', 'injuries_incapacitating',
'injuries_fatal', 'crash_hour', 'crash_day_of_week', 'crash_month'
]
crash = crash.drop(*columns_to_drop)

# Train/Test split
train_set, test_set = crash.randomSplit([0.7, 0.3], seed=42)

# Identify feature columns
target = "injury_flag"
feature_columns = [col for col in crash.columns if col != target]

# Automatically determine categorical and numeric columns
categorical_columns = [f.name for f in crash.schema.fields if str(f.dataType) == "StringType" and f.name in feature_columns]
numeric_columns = [f.name for f in crash.schema.fields if f.name in feature_columns and f.name not in categorical_columns]

# Index and encode categorical features
indexers = [StringIndexer(inputCol=col, outputCol=col + "_index", handleInvalid="keep") for col in categorical_columns]
encoders = [OneHotEncoder(inputCol=col + "_index", outputCol=col + "_encoded") for col in categorical_columns]

# Assemble feature vector
assembler_inputs = [col + "_encoded" for col in categorical_columns] + numeric_columns
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features_raw")

# 1. Confirm the list of numeric feature columns (no nulls!)
numeric_columns = ['night_crash', 'weekend_crash', 'rush_hour_crash', 
'multi_vehicle', 'any_incapacitating', 'any_fatalities']

# 2. Fill any nulls with zero — this is mandatory before VectorAssembler
crash = crash.fillna(0, subset=numeric_columns)

# 3. Assemble features into a single vector column
assembler = VectorAssembler(inputCols=numeric_columns, outputCol="features_raw")

from pyspark.ml.feature import StandardScaler

# 4. Apply scaler (this should work as long as features_raw is a Vector)
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withMean=True, withStd=True)

from pyspark.ml.classification import LogisticRegression

# Model
lr = LogisticRegression(labelCol=target, featuresCol="features", maxIter=1000)

# Pipeline
pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler, lr])

# Fit model
model = pipeline.fit(train_set)

# Predictions
train_pred = model.transform(train_set)
test_pred = model.transform(test_set)

# Evaluation

from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(
labelCol=target,
predictionCol="prediction",
metricName="f1"  
)

evaluator = MulticlassClassificationEvaluator(labelCol=target, predictionCol="prediction", metricName="accuracy")

train_accuracy = evaluator.evaluate(train_pred)
test_accuracy = evaluator.evaluate(test_pred)

print(f"Train Accuracy: {train_accuracy:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")


Train Accuracy: 0.7673
Test Accuracy: 0.7670


## ML Model 3

**Data Preparation:**

- Loads crash data.
- Extracts crash_day_of_week and crash_hour from crash_time.
- Removes rows with null values in crash_month, weather_condition, lighting_condition, and roadway_surface_cond.

**Feature Processing (Pipeline):**

- Encodes categorical features (weather_condition, lighting_condition, roadway_surface_cond) using String Indexing.
- Assembles the indexed categorical features, crash_day_of_week, and crash_hour into a single feature vector.
- Model Training and Evaluation:
- Splits the data into training (80%) and testing (20%) sets.
- Trains a Random Forest Classifier model.
- Evaluates the model's performance using accuracy and F1 score.
- Prints the accuracy and F1 score on the test data.
- Generates and displays a confusion matrix using Seaborn and Matplotlib.

**Results:**

Random Forest Test Accuracy = 0.1424
Random Forest Test F1 Score = 0.1069

**Interpretation:** The Random Forest Classifier model's performance in predicting the month of a crash is quite low. The accuracy of 0.1424 indicates that the model correctly predicts the crash month only about 14.24% of the time. Similarly, the F1 score of 0.1069, which considers both precision and recall, is also very low, suggesting poor predictive capability.

In [0]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

crash = crashmaster

# Feature Engineering: Extract crash_day_of_week and crash_hour
if "crash_time" in crash.columns:
    crash = crash.withColumn("crash_time", F.to_timestamp("crash_time"))
    crash = crash.withColumn("crash_day_of_week", F.dayofweek("crash_time"))  # 1 = Sunday
    crash = crash.withColumn("crash_hour", F.hour("crash_time"))

# Data Preprocessing: Drop rows with nulls in key columns
crash = crash.dropna(subset=["crash_month", "weather_condition", "lighting_condition", "roadway_surface_cond"])

# StringIndexing (encode) categorical features
weather_indexer = StringIndexer(inputCol="weather_condition", outputCol="weather_condition_index", handleInvalid="keep")
lighting_indexer = StringIndexer(inputCol="lighting_condition", outputCol="lighting_condition_index", handleInvalid="keep")
roadway_indexer = StringIndexer(inputCol="roadway_surface_cond", outputCol="roadway_surface_cond_index", handleInvalid="keep")

# Assemble all features into a single vector
assembler = VectorAssembler(
    inputCols=[
        "weather_condition_index",
        "lighting_condition_index",
        "roadway_surface_cond_index",
        "crash_day_of_week",
        "crash_hour"
    ],
    outputCol="features"
)

# Random Forest Model
rf = RandomForestClassifier(
    labelCol="crash_month",  
    featuresCol="features",
    numTrees=150,
    maxDepth=5,
    seed=42
)

# Create Pipeline
pipeline = Pipeline(stages=[weather_indexer, lighting_indexer, roadway_indexer, assembler, rf])

# Train-test split
train_data, test_data = crash.randomSplit([0.8, 0.2], seed=42)

# Fit model
model = pipeline.fit(train_data)

# Predict
predictions = model.transform(test_data)

# Evaluation Metrics
evaluator_acc = MulticlassClassificationEvaluator(labelCol="crash_month", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator_acc.evaluate(predictions)

evaluator_f1 = MulticlassClassificationEvaluator(labelCol="crash_month", predictionCol="prediction", metricName="f1")
f1_score = evaluator_f1.evaluate(predictions)

print(f"Random Forest Test Accuracy = {accuracy:.4f}")
print(f"Random Forest Test F1 Score = {f1_score:.4f}")




Random Forest Test Accuracy = 0.1424
Random Forest Test F1 Score = 0.1069


## ML Model 4

The code analyzes how weather conditions affect traffic crashes, specifically crash types and fatalities. It reads crash data, then defines a function that uses PySpark to:

**Data Preparation:**

- Reads crash data from a CSV file into a Spark DataFrame.

**Weather Condition vs. Crash Type Analysis:**

- Predicts crash types based on weather conditions using a Decision Tree Classifier.
- Evaluates the prediction accuracy.

**Results Interpretation:** The Decision Tree Classifier achieved a test accuracy of 0.5646, indicating limited predictive power. Weather conditions have some relationship with crash types, but other factors are more influential.

**Weather Condition vs. Fatalities Analysis:**

- Predicts the number of fatalities based on weather conditions using Linear Regression.
- Evaluates the model's R-squared value.

**Results Interpretation:** The Linear Regression model has an R-squared value of 0.0000. This means the model explains none of the variance in the number of fatalities. In simpler terms, weather conditions, as used in this model, do not predict the number of fatalities. The table shows that the model predicts very small fatality values (e.g., 2.627178699523144E-4) regardless of the weather condition, and the actual number of fatalities is often 0.

**Average Fatalities Calculation:**

Calculates and displays the average number of fatalities for each weather condition.

**Results Interpretation:** This table shows the average number of fatalities for each weather condition. 'SEVERE CROSS WINDS' has the highest average fatalities (0.03125), followed by 'FOG/SMOKE/HAZE'. Conditions like 'BLOWING SNOW', 'SLEET/HAIL', 'OTHER', and 'BLOWING SAND, SOIL, DIRT' have no fatalities in the data. It's important to note that these are average fatalities, and the actual number of fatalities in any given crash can vary.

In [0]:
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.regression import LinearRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, RegressionEvaluator

mastercrash = spark.read.csv("dbfs:/FileStore/shared_uploads/finalprojectgroup4040@gmail.com/export__1_.csv", header=True, inferSchema=True)

# Temp file
crash = mastercrash

def analyze_weather_impact(crash, train_ratio=0.7, seed=42):

    # 1. Weather Condition vs. Crash Type Analysis (Classification)
    print("\n--- 1. Weather Condition vs. Crash Type Analysis ---")

    # Prepare data for classification
    df_crash_type = crash.select("weather_condition", "crash_type").dropna()  # Remove nulls for this analysis

    # String Indexer for weather_condition and crash_type
    weather_indexer = StringIndexer(inputCol="weather_condition", outputCol="weather_index", handleInvalid="keep")
    crash_type_indexer = StringIndexer(inputCol="crash_type", outputCol="label", handleInvalid="keep")

    # Vector Assembler for features (weather_index)
    assembler_crash_type = VectorAssembler(inputCols=["weather_index"], outputCol="features")

    # Decision Tree Classifier
    dtc = DecisionTreeClassifier(featuresCol="features", labelCol="label", maxDepth=4)

    # Pipeline
    pipeline_crash_type = Pipeline(stages=[weather_indexer, crash_type_indexer, assembler_crash_type, dtc])

    # Train/test split
    train_crash_type, test_crash_type = df_crash_type.randomSplit([train_ratio, 1 - train_ratio], seed=seed)

    # Fit model
    model_crash_type = pipeline_crash_type.fit(train_crash_type)

    # Predict
    predictions_crash_type = model_crash_type.transform(test_crash_type)

    # Evaluate (Accuracy)
    evaluator_crash_type = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
    accuracy_crash_type = evaluator_crash_type.evaluate(predictions_crash_type)
    print(f"   Test Accuracy (Crash Type Prediction): {accuracy_crash_type:.4f}")

    # Show a sample of predictions
    predictions_crash_type.select("weather_condition", "crash_type", "prediction", "label").show(10)

    # 2. Weather Condition vs. Fatalities Analysis (Regression)
    print("\n--- 2. Weather Condition vs. Fatalities Analysis ---")

    # Prepare data for regression
    df_fatalities = crash.select("weather_condition", "injuries_fatal").dropna() # Remove nulls

    # String Indexer for weather condition
    weather_indexer_fatal = StringIndexer(inputCol="weather_condition", outputCol="weather_index", handleInvalid="keep")

    # Vector Assembler for features (weather_index)
    assembler_fatalities = VectorAssembler(inputCols=["weather_index"], outputCol="features") 

    # Linear Regression
    lr = LinearRegression(featuresCol="features", labelCol="injuries_fatal")

    # Pipeline
    pipeline_fatalities = Pipeline(stages=[weather_indexer_fatal, assembler_fatalities, lr]) 

    # Train/test split
    train_fatalities, test_fatalities = df_fatalities.randomSplit([train_ratio, 1 - train_ratio], seed=seed)

    # Fit model
    model_fatalities = pipeline_fatalities.fit(train_fatalities)

    # Predict
    predictions_fatalities = model_fatalities.transform(test_fatalities)

    # Evaluate (R-squared)
    evaluator_fatalities = RegressionEvaluator(labelCol="injuries_fatal", predictionCol="prediction", metricName="r2")
    r2_fatalities = evaluator_fatalities.evaluate(predictions_fatalities)
    print(f"   Test R-squared (Fatalities Prediction): {r2_fatalities:.4f}")

    # Show a sample of predictions
    predictions_fatalities.select("weather_condition", "injuries_fatal", "prediction").show(10)

    # Group by weather condition and show average fatalities
    crash.groupBy("weather_condition").agg(F.mean("injuries_fatal").alias("avg_fatalities")).orderBy("avg_fatalities", ascending=False).show()

# Assuming your cleaned DataFrame is named 'crash'
analyze_weather_impact(crash)


--- 1. Weather Condition vs. Crash Type Analysis ---
   Test Accuracy (Crash Type Prediction): 0.5646
+-----------------+--------------------+----------+-----+
|weather_condition|          crash_type|prediction|label|
+-----------------+--------------------+----------+-----+
|     BLOWING SNOW|INJURY AND / OR T...|       0.0|  1.0|
|     BLOWING SNOW|INJURY AND / OR T...|       0.0|  1.0|
|     BLOWING SNOW|INJURY AND / OR T...|       0.0|  1.0|
|     BLOWING SNOW|INJURY AND / OR T...|       0.0|  1.0|
|     BLOWING SNOW|NO INJURY / DRIVE...|       0.0|  0.0|
|     BLOWING SNOW|NO INJURY / DRIVE...|       0.0|  0.0|
|     BLOWING SNOW|NO INJURY / DRIVE...|       0.0|  0.0|
|     BLOWING SNOW|NO INJURY / DRIVE...|       0.0|  0.0|
|     BLOWING SNOW|NO INJURY / DRIVE...|       0.0|  0.0|
|     BLOWING SNOW|NO INJURY / DRIVE...|       0.0|  0.0|
+-----------------+--------------------+----------+-----+
only showing top 10 rows


--- 2. Weather Condition vs. Fatalities Analysis ---
   Te